In [1]:
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib as mpl
import matplotlib.pyplot as plt

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.mask import count, decode, variance
from bloodmoon.types import CoordEquatorial
from bloodmoon.optim import model_sky
import darksun as ds
from darksun.filtering import select_source_photons

from camgeom.simstudy import check_and_pick, get_source_coords

RCPARAMS = {
    'axes.formatter.limits': (-3, 3),
    'savefig.bbox': 'tight',
}
plt.rcdefaults()
mpl.rcParams.update(RCPARAMS)
ds.show.set_figures_darkbkg()

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "baseline_2-50keV_1ks"

ID_CAMERA: str = "cam1a"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]

UPS_X, UPS_Y = 2, 1
hide_bulk_els_y: float = 1.5   # [mm]

# BASEDIR: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations"
BASEDIR: str = "/mnt/d/PhD_AASS/Coding/Images_fits"
# DATAPATH: str = Path(f"{BASEDIR}/{SKYFIELD}/{DATA_FITS}")


wfm: CodedMaskCamera = codedmask(f'{BASEDIR}/{MASK_FITS}', UPS_X, UPS_Y, hide_bulk_els_y=hide_bulk_els_y)

# cat = ds.get_catalogue(check_and_pick(DATAPATH, f'{ID_CAMERA}/*sources*.fits'))
# sdl = ds.get_data(check_and_pick(DATAPATH, f'{ID_CAMERA}/*{DATASET}*.fits'), E_min=E_min, E_max=E_max)

In [61]:
assert False

AssertionError: 

In [ ]:
# srcID = 's17'

# mask = (cat.DLdata['NAME'] != srcID)
# exclude_srcs: list[CoordEquatorial] = [
#     CoordEquatorial(ra, dec) for ra, dec in zip(cat.DLdata['RA'][mask], cat.DLdata['DEC'][mask])
# ]
# phs = ds.filter_data(sdl.DLdata, E_min=None, E_max=None, coords=exclude_srcs)

# detector, _ = count(wfm, phs, rot_angle=rot_angle)
# skymap = decode(wfm, detector)
# varmap = variance(wfm, detector)

In [ ]:
# ra, dec = get_source_coords(srcID, cat)
# src_pos = equatorial2pos(sdl, wfm, ra, dec)
# src_peak = find_boxmax(skymap, src_pos, boxsize=2 * wfm.upscale_f.x + 1)

In [ ]:
# rows, cols = crop_source_psf(wfm, src_peak, 2 + wfm.upscale_f.x, 5 * wfm.upscale_f.y)
# psfvar = varmap[rows, cols]
# fit_weights = np.pow(np.clip(psfvar, a_min=1.0, a_max=None), -0.5).flatten()

In [ ]:
# from camgeom.simstudy import config_optimiser, config_fitter
# from camgeom.types import OptResult

# vignetting = True
# psfy = DATASET == "reconstructed"

# optimiser = config_optimiser(
#     camera=wfm,
#     vignetting=vignetting,
#     psfy=psfy,
#     fit_weights=fit_weights,
#     verbose=True,
# )
# fit_src_pars = config_fitter(optimiser)

In [ ]:
# pars, pcov = fit_src_pars(src_peak, skymap)

In [ ]:
# pars, np.sqrt(np.diag(pcov))

In [ ]:
def find_tick_val(ticks: NDArray, trg_vals: NDArray, atol: float = 1e-4) -> NDArray:
    """Finds the idxs inside `ticks` relative to given values `trg_vals`."""
    trg_idxs: list[int] = []
    for v in trg_vals:
        idx = np.argwhere(np.isclose(ticks, v, atol=atol))
        if idx.size > 0: trg_idxs.append(idx[0, 0])
    return np.array(trg_idxs)

def set_phase(x: NDArray) -> NDArray:
    """Centers `x` values around zero."""
    return np.arange(len(x) + 1) - len(x) // 2 - 0.5

def get_slices(*arrs: NDArray) -> tuple[tuple[NDArray, ...], tuple[NDArray, ...]]:
    n, m = arrs[0].shape
    xslice, yslice = zip(*tuple((c[(n - 1) // 2, :], c[:, (m - 1) // 2]) for c in arrs))
    return xslice, yslice

def get_heatmap_plot_data(
    psf: NDArray,
    psf_err: NDArray,
    modelled: NDArray,
    modelled_err: NDArray,
    mode: str,
    express_res_perc: bool,
) -> tuple[NDArray, str]:
    """Handles source PSF residuals heatmap to plot."""
    if mode == 'ratio':
        return psf / modelled, 'ratio'

    res = psf - modelled
    if mode == 'res':
        lbl = 'residuals [ph]'
        if express_res_perc:
            res = res * 100 / psf.max()
            lbl = r'residuals [%]'
        return res, lbl

    sigma = np.sqrt(np.square(psf_err) + np.square(modelled_err))
    return res / sigma, 'residuals [$\\sigma$]'

def get_slices_plot_data(
    psf: NDArray,
    psf_err: NDArray,
    modelled: NDArray,
    modelled_err: NDArray,
    mode: str,
    express_res_perc: bool,
    n_sigma: int,
) -> tuple[NDArray, NDArray, str]:
    """
    Handles data to plot from source PSF and sigma slices, for both true and modelled.
    """
    if mode not in ('res', 'ratio', 'sigma'):
        raise ValueError(f"Invalid input mode '{mode}'.")
    
    if mode == 'ratio':
        sigma = psf_err / np.abs(modelled) + np.abs(psf / np.square(modelled)) * modelled_err
        return psf / modelled, n_sigma * sigma, 'ratio'

    res = psf - modelled
    sigma = np.sqrt(np.square(psf_err) + np.square(modelled_err))

    if mode == 'res':
        lbl = 'residuals [counts]'
        if express_res_perc:
            res, sigma = map(lambda x: x * 100 / psf.max(), (res, sigma))
            lbl = r'residuals [%]'
        return res, n_sigma * sigma, lbl

    return res / sigma, n_sigma * np.ones_like(res), 'residuals [$\\sigma$]'

def show_PSF_total(
    camera: CodedMaskCamera,
    true_w_err: tuple[NDArray, NDArray],
    modelled_w_err: tuple[NDArray, NDArray],
    save_to: str | None = None,
    express_res_perc: bool = True,
    express_res_sigma: bool = False,
    n_sigma: int = 1,
    cts_lims: tuple = (None, None),
    hm_ratio_lims: tuple = (None, None),
    fine_ratio_lims: tuple = (None, None),
    coarse_ratio_lims: tuple = (None, None),
    save_options: dict | None = None,
    **fig_kws,
) -> None:
    """
    Plots the source PSF heat maps and profiles along fine/coarse axes + cfr with modelled source.
    """
    res_mode = 'res'
    if express_res_sigma:
        express_res_perc = False
        res_mode = 'sigma'

    title_fs = 16
    lbl_fs = 14
    
    dtx, dty = ds.pixels_angular_resolution(camera, verbose=False)
    psf, modelled = true_w_err[0], modelled_w_err[0]
    psf_xslice, psf_yslice = get_slices(*true_w_err)
    modelled_xslice, modelled_yslice = get_slices(*modelled_w_err)

    phase_fine, phase_coarse = (
        dtx * set_phase(psf_xslice[0]),
        dty * set_phase(psf_yslice[0]) / 60,
    )
    phase_ferr, phase_cerr = (
        dtx * (set_phase(psf_xslice[0])[:-1] + 0.5),
        dty * (set_phase(psf_yslice[0])[:-1] + 0.5) / 60,
    )
    
    slice_clrs = ('OrangeRed', 'DodgerBlue')
    img_factory = dict(origin='lower', interpolation='none', aspect=2 * psf.shape[1] / psf.shape[0])
    errbar_factory = dict(linestyle='', elinewidth=0.7, capsize=0)

    fig, axs = plt.subplots(3, 3, tight_layout=True, **fig_kws)
    axs = axs.flatten()

    # ------------------------ 1st ROW (PSF HM + Profiles f/c) ---------------------------
    slice_lim_vals = max(modelled.max(), psf.max())
    slice_lim_vals = (cts_lims[0] or -0.075 * slice_lim_vals, cts_lims[1] or 1.05 * slice_lim_vals)
    
    axs[0].set_title('Crab PSF', fontsize=title_fs)
    psf_img = axs[0].imshow(psf, cmap='gist_heat', vmin=cts_lims[0], vmax=cts_lims[1], **img_factory)
    cbar = fig.colorbar(psf_img, ax=axs[0], location='right', shrink=0.8, pad=0.05)
    cbar.set_label('counts', fontsize=lbl_fs)
    cbar.ax.tick_params(labelsize=lbl_fs)
    axs[0].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)
    axs[0].set_ylabel(ylabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)

    axs[1].set_title('PSF Slice - Fine Axis', fontsize=title_fs)
    axs[1].stairs(psf_xslice[0], edges=phase_fine, edgecolor=slice_clrs[0], label='true')
    axs[1].stairs(modelled_xslice[0], edges=phase_fine, edgecolor=slice_clrs[1], label='modelled')
    axs[1].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)

    axs[2].set_title('PSF Slice - Coarse Axis', fontsize=title_fs)
    axs[2].stairs(psf_yslice[0], edges=phase_coarse, edgecolor=slice_clrs[0], label='true')
    axs[2].stairs(modelled_yslice[0], edges=phase_coarse, edgecolor=slice_clrs[1], label='modelled')
    axs[2].set_xlabel(xlabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)
    axs[2].set_ylabel(ylabel='counts', labelpad=10, fontsize=lbl_fs)

    for idx in [1, 2]: axs[idx].set_ylim(slice_lim_vals)

    # ------------------------ 2nd ROW (RES [%] HM + Profiles f/c) ------------------------    
    psf_hm_res, hm_lbl = get_heatmap_plot_data(*true_w_err, *modelled_w_err, res_mode, express_res_perc)
    xres, xsigma, lbl = get_slices_plot_data(*psf_xslice, *modelled_xslice, res_mode, express_res_perc, n_sigma)
    yres, ysigma, _ = get_slices_plot_data(*psf_yslice, *modelled_yslice, res_mode, express_res_perc, n_sigma)

    axs[3].set_title(f'Crab PSF Residuals', fontsize=title_fs)
    mod_img = axs[3].imshow(psf_hm_res, **img_factory)
    cbar = fig.colorbar(mod_img, ax=axs[3], location='right', shrink=0.8, pad=0.05)
    cbar.set_label(hm_lbl, fontsize=lbl_fs)
    cbar.ax.tick_params(labelsize=lbl_fs)
    axs[3].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)
    axs[3].set_ylabel(ylabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)

    axs[4].set_title(f'PSF Slice Residuals - Fine Axis', fontsize=title_fs)
    axs[4].plot(phase_fine, np.zeros_like(phase_fine), slice_clrs[0], label='ground', linestyle='--', alpha=0.6)
    axs[4].stairs(xres, edges=phase_fine, edgecolor=slice_clrs[1], label=lbl)
    axs[4].errorbar(phase_ferr, xres, c=slice_clrs[1], yerr=xsigma, marker='', **errbar_factory)
    axs[4].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)

    axs[5].set_title(f'PSF Slice Residuals - Coarse Axis', fontsize=title_fs)
    axs[5].plot(phase_coarse, np.zeros_like(phase_coarse), slice_clrs[0], label='ground', linestyle='--', alpha=0.6)
    axs[5].stairs(yres, edges=phase_coarse, edgecolor=slice_clrs[1], label=lbl)
    axs[5].errorbar(phase_cerr, yres, c=slice_clrs[1], yerr=ysigma, marker='', **errbar_factory)
    axs[5].set_xlabel(xlabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)
    axs[5].set_ylabel(ylabel=lbl + f" (C.L. ${n_sigma}\\sigma$)", labelpad=10, fontsize=lbl_fs)

    # ------------------------ 3rd ROW (RATIO HM + Profiles f/c) --------------------------
    psf_hm_res, hm_lbl = get_heatmap_plot_data(*true_w_err, *modelled_w_err, 'ratio', express_res_perc)
    xres, xsigma, lbl = get_slices_plot_data(*psf_xslice, *modelled_xslice, 'ratio', express_res_perc, n_sigma)
    yres, ysigma, _ = get_slices_plot_data(*psf_yslice, *modelled_yslice, 'ratio', express_res_perc, n_sigma)

    vlim = max(np.abs(xres).max(), np.abs(yres).max())
    hm_res_lims_ = (hm_ratio_lims[0] or -vlim + 1, hm_ratio_lims[1] or vlim + 1)

    axs[6].set_title(f'Crab PSF Ratio', fontsize=title_fs)
    mod_img = axs[6].imshow(psf_hm_res, vmin=hm_res_lims_[0], vmax=hm_res_lims_[1], **img_factory)
    cbar = fig.colorbar(mod_img, ax=axs[6], location='right', shrink=0.8, pad=0.05)
    cbar.set_label(hm_lbl, fontsize=lbl_fs)
    cbar.ax.tick_params(labelsize=lbl_fs)
    axs[6].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)
    axs[6].set_ylabel(ylabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)

    axs[7].set_title(f'PSF Slice Ratio - Fine Axis', fontsize=title_fs)
    axs[7].plot(phase_fine, np.ones_like(phase_fine), slice_clrs[0], label='ground', linestyle='--', alpha=0.6)
    axs[7].stairs(xres, edges=phase_fine, edgecolor=slice_clrs[1], label=lbl)
    axs[7].errorbar(phase_ferr, xres, c=slice_clrs[1], yerr=xsigma, marker='', **errbar_factory)
    axs[7].set_ylim(fine_ratio_lims)
    axs[7].set_xlabel(xlabel='$\\Delta\\theta_{fine}$ [arcmin]', fontsize=lbl_fs)

    axs[8].set_title(f'PSF Slice Ratio - Coarse Axis', fontsize=title_fs)
    axs[8].plot(phase_coarse, np.ones_like(phase_coarse), slice_clrs[0], label='ground', linestyle='--', alpha=0.6)
    axs[8].stairs(yres, edges=phase_coarse, edgecolor=slice_clrs[1], label=lbl)
    axs[8].errorbar(phase_cerr, yres, c=slice_clrs[1], yerr=ysigma, marker='', **errbar_factory)
    axs[8].set_ylim(coarse_ratio_lims)
    axs[8].set_xlabel(xlabel='$\\Delta\\theta_{coarse}$ [deg]', fontsize=lbl_fs)
    axs[8].set_ylabel(ylabel=lbl + f" (C.L. ${n_sigma}\\sigma$)", labelpad=10, fontsize=lbl_fs)

    # ticks for HMs
    yticks, xticks = map(lambda s: np.arange(0, s), psf.shape)
    _ytick_lbls, _xtick_lbls = (
        dty / 60 * (yticks - len(yticks) // 2), dtx * (xticks - len(xticks) // 2),
    )
    yticks_ = find_tick_val(_ytick_lbls, np.array([-5.0, -2.5, 0.0, 2.5, 5.0]), atol=7e-2)
    yticks_lbl = [float(f'{t:.1f}') for t in _ytick_lbls[yticks_]]
    xticks_ = find_tick_val(_xtick_lbls, np.array([-5.0, 0.0, 5.0]), atol=1e-1)
    xticks_lbl = [float(f'{t:.1f}') for t in _xtick_lbls[xticks_]]

    for idx in [0, 3, 6]:
        axs[idx].set_xticks(xticks_, xticks_lbl)
        axs[idx].set_yticks(yticks_, yticks_lbl)
    
    for idx in [1, 2, 4, 5, 7, 8]:
        axs[idx].yaxis.tick_right()
        axs[idx].legend(loc='best')

    for idx in [2, 5, 8]:
        axs[idx].yaxis.set_label_position('right')

    for idx, ax in enumerate(axs):
        ax.xaxis.set_ticks_position('both')
        ax.yaxis.set_ticks_position('both')
        ax.tick_params(which='both', direction='in', width=1.5, length=4 if 'major' else 2.5, labelsize=lbl_fs)

    if save_to is not None:
        save_options_ = save_options or {}
        fig.savefig(save_to, **save_options_)
    
    plt.show()
    return

In [ ]:
# sky_model = model_sky(wfm, *pars, vignetting=vignetting, psfy=psfy)
# var_model = np.zeros_like(sky_model)

# get_cropped = lambda x: ds.crop(x, src_peak, (38 * UPS_Y, 2 * (UPS_X - 1) + 1))
# psf_true, psf_var = map(get_cropped, (skymap, varmap))
# psf_model, psf_model_var = map(get_cropped, (sky_model, var_model))

# show_PSF_total(
#     camera=wfm,
#     true_w_err=(psf_true, np.sqrt(psf_var)),
#     modelled_w_err=(psf_model, np.zeros_like(psf_model)),  # NOTE: [validate sg error consistency] np.sqrt(psf_model_var)
#     express_res_perc=True,
#     # express_res_sigma=True,
#     n_sigma=1,
#     figsize=(4 * 3 + 0.5, 4 * 3 + 0.5),
# )

In [ ]:
from typing import Callable, NamedTuple

from scipy.optimize import least_squares

from bloodmoon.coords import pos2shift
from darksun.benchmarking import crop_source_psf

class NewOptResult(NamedTuple):
    """Optimisation results."""
    params: NDArray
    covar: NDArray
    red_chi2: float
    status: int

def updated_fit_source_psf(
    camera: CodedMaskCamera,
    sky: NDArray,
    arg_sky: tuple[int, int],
    sigma: NDArray,
    vignetting: bool | Callable[..., NDArray],
    psfy: bool | Callable[..., NDArray],
    cam_coding_power: float = 0.85,
    psf_crp_xyoff: tuple[int, int] = (0, 0),
    shift_box: tuple[float, float] | None = None,
) -> NewOptResult:
    rows, cols = crop_source_psf(camera, arg_sky, *psf_crp_xyoff)
    ydata = sky[rows, cols].ravel()
    inv_sigma = 1.0 / np.asarray(sigma, dtype=np.float64).ravel()
    sky_peak = float(sky[*arg_sky])
    if sky_peak <= 0:
        raise ValueError(f"Non-positive sky peak ({sky_peak:.3g}) at {arg_sky}.")

    sx0, sy0 = pos2shift(camera, *arg_sky)
    cts0 = sky_peak / cam_coding_power if psfy else sky_peak

    dx, dy = shift_box or (
        camera.specs.mask_deltax / camera.upscale_f.x,
        camera.specs.mask_deltay / camera.upscale_f.y,
    )
    lb = np.array([
        max(sx0 - dx, float(camera.bins_sky.x[0])),
        max(sy0 - dy, float(camera.bins_sky.y[0])),
        0.95 * sky_peak,
    ])
    ub = np.array([
        min(sx0 + dx, float(camera.bins_sky.x[-1])),
        min(sy0 + dy, float(camera.bins_sky.y[-1])),
        max(1.4 * sky_peak, 1.2 * cts0),
    ])
    p0 = np.clip(np.array([sx0, sy0, cts0], dtype=np.float64), lb, ub)

    def unit_model(sx: float, sy: float) -> NDArray:
        return model_sky(camera, sx, sy, 1.0, vignetting, psfy)[rows, cols].ravel()

    # Cache last (sx, sy) -> M1 so jac reuses the residual's unit_model call
    cache: dict[tuple[float, float], NDArray] = {}

    def get_unit(sx: float, sy: float) -> NDArray:
        key = (sx, sy)
        if key not in cache:
            cache.clear()
            cache[key] = unit_model(sx, sy)
        return cache[key]

    def residuals(p: NDArray) -> NDArray:
        sx, sy, fluence = float(p[0]), float(p[1]), float(p[2])
        return (fluence * get_unit(sx, sy) - ydata) * inv_sigma

    # Step size scaled to sub-pixel geometry (avoiding tiny steps near sx=0)
    h_x, h_y = 1e-4 * dx, 1e-4 * dy

    def jac(p: NDArray) -> NDArray:
        sx, sy, fluence = float(p[0]), float(p[1]), float(p[2])
        m0 = get_unit(sx, sy)
        # Forward/backward diff stays strictly within bounds
        hx = h_x if sx + h_x <= ub[0] else -h_x
        hy = h_y if sy + h_y <= ub[1] else -h_y
        dm_dsx = (unit_model(sx + hx, sy) - m0) / hx
        dm_dsy = (unit_model(sx, sy + hy) - m0) / hy
        J = np.empty((ydata.size, 3), dtype=np.float64)
        J[:, 0] = fluence * dm_dsx * inv_sigma
        J[:, 1] = fluence * dm_dsy * inv_sigma
        J[:, 2] = m0 * inv_sigma
        return J

    res = least_squares(
        residuals,
        x0=p0,
        jac=jac,
        bounds=(lb, ub),
        method='trf',
        ftol=1e-8,
        xtol=1e-8,
        x_scale='jac',
    )

    dof = max(ydata.size - p0.size, 1)
    red_chi2 = float(2.0 * res.cost / dof)
    # Covariance via SVD of Jacobian (matches scipy.optimize.curve_fit, robust to rank deficiency)
    _, s, vt = np.linalg.svd(res.jac, full_matrices=False)
    thresh = np.finfo(float).eps * max(res.jac.shape) * s[0]
    s_inv = np.where(s > thresh, 1.0 / s, 0.0)
    pcov = (vt.T * (s_inv ** 2)) @ vt  # multiply by red_chi2 if you want relative covariance

    return NewOptResult(res.x, pcov, red_chi2, int(res.status))

In [ ]:
from camgeom.simstudy import analyse_camgeom, load_from_csv

SKYFIELD: str = "CamGeom"
DATA_FITS: str = "baseline/baseline"
DATAPATH: str = Path(f"{BASEDIR}/{SKYFIELD}/{DATA_FITS}")

analyse_camgeom(
    in_out_fpaths=(DATAPATH, f'{BASEDIR}/src_out_df_test.csv'),
    camera=wfm,
    energy_band=(E_min, E_max),
    vignetting=True,
    psfy=True,
    psf_crp_xyoff=(2 + wfm.upscale_f.x, 5 * wfm.upscale_f.y),
    src_optim_kws=dict(verbose=True),
    phs_corr=None,
)

CATALOG ='camera_benchmark_catalogue.fits' / Catalog file                        [astropy.io.fits.card]


UserInfo: using bulk mask of [0.0 x 1.5] mm.


ID: S1:  11%|█         | 1/9 [00:08<01:09,  8.66s/it]    


## Optimisation Results:
  * p[0]:
      - START.: -142.1250000
      - OPTIM.: -142.1973233 +/- 0.0006116
      - GAIN.: -0.051 %
  * p[1]:
      - START.: -142.0000000
      - OPTIM.: -141.9385315 +/- 0.0287997
      - GAIN.: 0.043 %
  * p[2]:
      - START.: 22355.4819401
      - OPTIM.: 21567.1287991 +/- 77.6459496
      - GAIN.: -3.526 %

## Fit Report:
  - func calls (also # of iters): 7
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values



ID: S2:  22%|██▏       | 2/9 [00:15<00:52,  7.44s/it]


## Optimisation Results:
  * p[0]:
      - START.: 142.1250000
      - OPTIM.: 142.1992927 +/- 0.0006166
      - GAIN.: 0.052 %
  * p[1]:
      - START.: -142.0000000
      - OPTIM.: -142.3727254 +/- 0.0237077
      - GAIN.: -0.262 %
  * p[2]:
      - START.: 22115.1774050
      - OPTIM.: 21059.7042918 +/- 70.1840359
      - GAIN.: -4.773 %

## Fit Report:
  - func calls (also # of iters): 6
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values



... (10 outputs left out)

ID: S8:  89%|████████▉ | 8/9 [00:57<00:07,  7.01s/it]


## Optimisation Results:
  * p[0]:
      - START.: -54.3750000
      - OPTIM.: -54.4212011 +/- 0.0001735
      - GAIN.: -0.085 %
  * p[1]:
      - START.: 54.5000000
      - OPTIM.: 54.4188389 +/- 0.0101370
      - GAIN.: -0.149 %
  * p[2]:
      - START.: 152867.7623942
      - OPTIM.: 146231.7367571 +/- 173.2975808
      - GAIN.: -4.341 %

## Fit Report:
  - func calls (also # of iters): 6
  - procedure msg: `ftol` termination condition is satisfied.
  - success status 2: Convergence in parameter values



ID: S8: 100%|██████████| 9/9 [01:13<00:00,  8.13s/it]


## Optimisation Results:
  * p[0]:
      - START.: 0.0000000
      - OPTIM.: 0.0006179 +/- 0.0001405
      - GAIN.: 0.001 %
  * p[1]:
      - START.: 0.0000000
      - OPTIM.: 0.0000034 +/- 0.0064498
      - GAIN.: 0.000 %
  * p[2]:
      - START.: 185757.3317580
      - OPTIM.: 169787.6124731 +/- 154.2911321
      - GAIN.: -8.597 %

## Fit Report:
  - func calls (also # of iters): 23
  - procedure msg: `xtol` termination condition is satisfied.
  - success status 3: Convergence in both chi-square and parameter values



In [ ]:
data = load_from_csv(f'{BASEDIR}/src_out_df_test.csv')
data

,srcID,shiftX_opt [mm],shiftY_opt [mm],angleX [deg],angleY [deg],DangleX [arcmin],DangleY [arcmin],true cts,optim cts,Dcts [%],D_Dcts [%],Dcts [sigma],cc peak cts,cc Dcts [%],cc Dcts [sigma],SNR
0,s0,-142.197323,-141.938532,-3.500000e+01,-34.999998,-0.229504,2.712115,22144.0,21567.128799,-2.605090,1.413625,-1.842845,19002.159649,-14.188224,-10.360550,62.661626
1,s1,142.199293,-142.372725,3.500000e+01,-35.000010,0.251598,-2.220334,22172.0,21059.704292,-5.016668,1.404008,-3.573105,18797.900794,-15.217839,-11.125295,61.981638
2,s2,142.204215,142.174132,3.500000e+01,35.000007,0.307520,-0.034593,22081.0,20812.448261,-5.744992,1.423074,-4.037030,18079.665352,-18.121166,-13.195725,59.623678
3,s3,-142.203400,142.335489,-3.499999e+01,34.999996,-0.298630,1.798501,22374.0,21790.534554,-2.607783,1.383368,-1.885098,19380.109851,-13.381113,-9.916698,64.192971
4,s4,-54.421349,-54.437284,-1.500000e+01,-15.000006,-0.225265,-0.476682,149502.0,148282.583603,-0.815652,0.477273,-1.708984,131852.555118,-11.805491,-25.273672,188.810374
5,s5,54.421344,-54.444056,1.500000e+01,-15.000003,0.225248,-0.583826,148751.0,146254.009109,-1.678638,0.480846,-3.491008,130054.815649,-12.568779,-26.788544,186.347068
6,s6,54.421569,54.399130,1.500000e+01,15.000000,0.228796,-0.125649,149020.0,148086.967078,-0.626113,0.477782,-1.310456,130937.450727,-12.134310,-25.904483,187.576812
7,s7,-54.421201,54.418839,-1.500000e+01,15.000004,-0.222958,0.185468,148502.0,146231.736757,-1.528776,0.484147,-3.157672,129937.598035,-12.501112,-26.605359,186.218574
8,s8,0.000618,0.000003,-5.340578e-07,0.000005,0.010493,-0.000263,171755.0,169787.612473,-1.145462,0.425546,-2.691745,157893.731994,-8.070372,-19.401968,221.007857


In [ ]:
# NOTE: v1 with simple src filtering logic (v2 on camgeom.simstudy has CXB pre-filter + per-source filtering)

# def fit_src_in_skygrid(
#     camera: CodedMaskCamera,
#     sdl: DataLoader,
#     ctg: CatalogueLoader,
#     config_optim: Callable[[NDArray], Callable[[NDArray, tuple[int, int]], OptResult]],
#     psf_crp_xyoff: tuple[int, int] = (0, 0),
#     **phs_corr,
# ) -> dict[str, dict[str, float]]:
#     """
#     Doc...
#     """
#     out_dmap: dict[str, dict[str, float]] = {}
#     src_list = ctg.DLdata[(ctg.DLdata['ID'] != '-1')]  # exclude CXB

#     loop = tqdm(src_list['NAME'], desc="Sky-field Analysis")
#     for srcID in loop:
#         loop.set_description_str(f'ID: {srcID.upper()}')
        
#         # filter for source + CAI operations
#         src_mask = (src_list != srcID)
#         exclude_srcs: list[CoordEquatorial] = [
#             CoordEquatorial(ra, dec) for ra, dec in zip(src_list['RA'][src_mask], src_list['DEC'][src_mask])
#         ]
#         phs = ds.filter_data(sdl.DLdata, E_min=None, E_max=None, coords=exclude_srcs)

#         detector, _ = count(camera, phs, **phs_corr)
#         skymap = decode(camera, detector)
#         varmap = variance(camera, detector)

#         # fit source params from PSF
#         src_coords = get_source_coords(srcID, ctg)
#         src_pos = equatorial2pos(sdl, camera, *src_coords)
#         src_peak = find_boxmax(skymap, src_pos, boxsize=2 * camera.upscale_f.x + 1)

#         rows, cols = crop_source_psf(camera, src_peak, *psf_crp_xyoff)
#         psfvar = varmap[rows, cols]
#         fit_weights = np.pow(np.clip(psfvar, a_min=1.0, a_max=None).ravel(), -0.5)

#         optimiser = config_optim(fit_weights=fit_weights)
#         fit_src_pars = config_fitter(optimiser)
#         pars, pcov = fit_src_pars(src_peak, skymap)
#         sx_opt, sy_opt, cts_opt = pars

#         # compute out vals
#         tx_true, ty_true = equatorial2angle(sdl, camera, *src_coords)
#         tx_opt, ty_opt = map(lambda x: shift2angle(camera, x), (sx_opt, sy_opt))
        
#         cts_peak, var_peak = skymap[*src_peak], varmap[*src_peak]
#         peak_sigma = pow(var_peak, 0.5)
#         tot_sigma = pow(var_peak + pcov[-1, -1], 0.5)

#         src_phs = select_source_photons(src_coords, phs, verbose=False)
#         cts_true = count(camera, src_phs, **phs_corr)[0].sum()

#         out_dmap[srcID] = {
#             'shiftX_opt [mm]': sx_opt,
#             'shiftY_opt [mm]': sy_opt,

#             'angleX [deg]': tx_true,
#             'angleY [deg]': ty_true,
#             'DangleX [arcmin]': (tx_opt - tx_true) * 60,
#             'DangleY [arcmin]': (ty_opt - ty_true) * 60,

#             'true cts': cts_true,
#             'optim cts': cts_opt,
#             'Dcts [%]': (cts_opt - cts_true) * 100 / cts_true,
#             'D_Dcts [%]': tot_sigma * 100 / cts_true,
#             'Dcts [sigma]': (cts_opt - cts_true) / tot_sigma,

#             'cc peak cts': cts_peak,
#             'cc Dcts [%]': (cts_peak - cts_true) * 100 / cts_true,
#             'cc Dcts [sigma]': (cts_peak - cts_true) / peak_sigma,

#             'SNR': cts_peak / peak_sigma,
#         }
#         for a in out_dmap[srcID]:
#             out_dmap[srcID][a] = float(out_dmap[srcID][a])

#     return out_dmap